# I. Import thư viện

In [77]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')                     
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, roc_auc_score,
                             roc_curve, confusion_matrix)
from sklearn.preprocessing import StandardScaler

# II. Xử lý tiền dữ liệu

## 1. Tổng hợp các hàm sử dụng

In [78]:
# Kiểm tra dữ liệu
def check_data_quality(df):
    print("="*50)
    print(" BÁO CÁO CHẤT LƯỢNG DỮ LIỆU ".center(50))
    print("="*50)
    
    # ---------------------------------------------------------
    # 1. KIỂM TRA MISSING VALUES
    # ---------------------------------------------------------
    print("\n1. MISSING VALUES (Giá trị thiếu):")
    missing_count = df.isnull().sum()
    missing_pct = (missing_count / len(df)) * 100
    missing_df = pd.DataFrame({'Total Missing': missing_count, 'Percentage (%)': missing_pct})
    missing_df = missing_df[missing_df['Total Missing'] > 0].sort_values(by='Total Missing', ascending=False)
    
    if not missing_df.empty:
        print(missing_df.round(2))
    else:
        print(" -> Không có giá trị nào bị thiếu.")

    # ---------------------------------------------------------
    # 2. KIỂM TRA OUTLIERS (Sử dụng phương pháp IQR)
    # ---------------------------------------------------------
    print("\n2. OUTLIERS (Giá trị ngoại lai trong các cột số):")
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    outlier_summary = {}
    
    if len(numeric_cols) > 0:
        for col in numeric_cols:
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower_bound = Q1 - 1.5 * IQR
            upper_bound = Q3 + 1.5 * IQR
            
            # Đếm số lượng outliers
            outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
            if len(outliers) > 0:
                outlier_summary[col] = len(outliers)
                
        if outlier_summary:
            for col, count in outlier_summary.items():
                print(f" -> Cột '{col}': Phát hiện {count} outliers ({(count/len(df))*100:.2f}%)")
        else:
            print(" -> Không phát hiện outliers nào dựa trên quy tắc 1.5*IQR.")
    else:
        print(" -> Không có cột dữ liệu dạng số để kiểm tra outliers.")

    # ---------------------------------------------------------
    # 3. KIỂM TRA INCONSISTENCIES
    # ---------------------------------------------------------
    print("\n3. INCONSISTENCIES (Dữ liệu không nhất quán):")
    
    # A. Dữ liệu trùng lặp (Duplicates)
    duplicates = df.duplicated().sum()
    print(f" -> Dữ liệu trùng lặp (Duplicated Rows): {duplicates} dòng")
    
    # B. Dữ liệu hỗn hợp hoặc kiểu dữ liệu bất thường
    print("\n -> Kiểm tra các cột phân loại (Categorical/Object):")
    object_cols = df.select_dtypes(include=['object', 'category']).columns
    if len(object_cols) > 0:
        for col in object_cols:
            unique_vals = df[col].nunique()
            print(f"    - Cột '{col}' có {unique_vals} giá trị duy nhất.")
            # Hiển thị các giá trị nếu có ít biến thể (dễ phát hiện lỗi gõ sai như 'Male', 'male', 'M')
            if unique_vals <= 15:
                print(f"      Chi tiết: {df[col].dropna().unique().tolist()}")
    else:
         print("    - Không có cột dạng chuỗi (object/categorical) để kiểm tra.")
            
    print("\n" + "="*50)

# Kiểm tra mã không hợp lệ   
def detect_undocumented_categories(df, expected_categories):
    """
    df: DataFrame cần kiểm tra.
    expected_categories: Dictionary định nghĩa các cột và danh sách giá trị hợp lệ của chúng.
    """
    print("="*60)
    print(" BÁO CÁO MÃ KHÔNG HỢP LỆ (UNDOCUMENTED CATEGORIES) ".center(60))
    print("="*60)
    
    found_issues = False
    
    for col, valid_values in expected_categories.items():
        if col in df.columns:
            # Lấy tập hợp các giá trị thực tế trong dữ liệu (bỏ qua giá trị NaN)
            actual_values = set(df[col].dropna().unique())
            valid_set = set(valid_values)
            
            # Tìm các giá trị có trong thực tế nhưng KHÔNG nằm trong danh sách hợp lệ
            invalid_values = actual_values - valid_set
            
            if invalid_values:
                found_issues = True
                print(f"\n[!] CỘT '{col}': Phát hiện dữ liệu ngoài danh mục!")
                print(f"    - Danh mục hợp lệ (chuẩn): {valid_values}")
                print(f"    - Mã không hợp lệ tìm thấy: {list(invalid_values)}")
                
                # Đếm số lần xuất hiện của các mã không hợp lệ
                invalid_counts = df[df[col].isin(invalid_values)][col].value_counts()
                print(f"    - Chi tiết số lượng:")
                for val, count in invalid_counts.items():
                    print(f"        + Mã '{val}' xuất hiện {count} lần")
        else:
            print(f"\n[?] Cảnh báo: Cột '{col}' không tồn tại trong DataFrame này.")
            
    if not found_issues:
        print("\n -> Tuyệt vời! Không phát hiện mã nào ngoài danh mục khai báo.")
        
    print("\n" + "="*60)

## 2. Đọc dữ liệu

In [79]:
df = pd.read_csv(".\default of credit card clients.csv",header = 1)
df

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,1,20000,2,2,1,24,2,2,-1,-1,...,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,4,50000,2,2,1,37,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,5,50000,1,2,1,57,-1,0,-1,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29995,29996,220000,1,3,1,39,0,0,0,0,...,88004,31237,15980,8500,20000,5003,3047,5000,1000,0
29996,29997,150000,1,3,2,43,-1,-1,-1,-1,...,8979,5190,0,1837,3526,8998,129,0,0,0
29997,29998,30000,1,2,2,37,4,3,2,-1,...,20878,20582,19357,0,0,22000,4200,2000,3100,1
29998,29999,80000,1,3,1,41,1,-1,0,0,...,52774,11855,48944,85900,3409,1178,1926,52964,1804,1


In [80]:
df.describe()

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
count,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,...,30000.000000,30000.000000,30000.000000,30000.000000,3.000000e+04,30000.00000,30000.000000,30000.000000,30000.000000,30000.000000
mean,15000.500000,167484.322667,1.603733,1.853133,1.551867,35.485500,-0.016700,-0.133767,-0.166200,-0.220667,...,43262.948967,40311.400967,38871.760400,5663.580500,5.921163e+03,5225.68150,4826.076867,4799.387633,5215.502567,0.221200
std,8660.398374,129747.661567,0.489129,0.790349,0.521970,9.217904,1.123802,1.197186,1.196868,1.169139,...,64332.856134,60797.155770,59554.107537,16563.280354,2.304087e+04,17606.96147,15666.159744,15278.305679,17777.465775,0.415062
min,1.000000,10000.000000,1.000000,0.000000,0.000000,21.000000,-2.000000,-2.000000,-2.000000,-2.000000,...,-170000.000000,-81334.000000,-339603.000000,0.000000,0.000000e+00,0.00000,0.000000,0.000000,0.000000,0.000000
25%,7500.750000,50000.000000,1.000000,1.000000,1.000000,28.000000,-1.000000,-1.000000,-1.000000,-1.000000,...,2326.750000,1763.000000,1256.000000,1000.000000,8.330000e+02,390.00000,296.000000,252.500000,117.750000,0.000000
50%,15000.500000,140000.000000,2.000000,2.000000,2.000000,34.000000,0.000000,0.000000,0.000000,0.000000,...,19052.000000,18104.500000,17071.000000,2100.000000,2.009000e+03,1800.00000,1500.000000,1500.000000,1500.000000,0.000000
75%,22500.250000,240000.000000,2.000000,2.000000,2.000000,41.000000,0.000000,0.000000,0.000000,0.000000,...,54506.000000,50190.500000,49198.250000,5006.000000,5.000000e+03,4505.00000,4013.250000,4031.500000,4000.000000,0.000000
max,30000.000000,1000000.000000,2.000000,6.000000,3.000000,79.000000,8.000000,8.000000,8.000000,8.000000,...,891586.000000,927171.000000,961664.000000,873552.000000,1.684259e+06,896040.00000,621000.000000,426529.000000,528666.000000,1.000000


In [81]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 25 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   ID                          30000 non-null  int64
 1   LIMIT_BAL                   30000 non-null  int64
 2   SEX                         30000 non-null  int64
 3   EDUCATION                   30000 non-null  int64
 4   MARRIAGE                    30000 non-null  int64
 5   AGE                         30000 non-null  int64
 6   PAY_0                       30000 non-null  int64
 7   PAY_2                       30000 non-null  int64
 8   PAY_3                       30000 non-null  int64
 9   PAY_4                       30000 non-null  int64
 10  PAY_5                       30000 non-null  int64
 11  PAY_6                       30000 non-null  int64
 12  BILL_AMT1                   30000 non-null  int64
 13  BILL_AMT2                   30000 non-null  int64
 14  BILL_A

## 3. Xử lý

In [82]:
check_data_quality(df)

            BÁO CÁO CHẤT LƯỢNG DỮ LIỆU            

1. MISSING VALUES (Giá trị thiếu):
 -> Không có giá trị nào bị thiếu.

2. OUTLIERS (Giá trị ngoại lai trong các cột số):
 -> Cột 'LIMIT_BAL': Phát hiện 167 outliers (0.56%)
 -> Cột 'EDUCATION': Phát hiện 454 outliers (1.51%)
 -> Cột 'AGE': Phát hiện 272 outliers (0.91%)
 -> Cột 'PAY_0': Phát hiện 3130 outliers (10.43%)
 -> Cột 'PAY_2': Phát hiện 4410 outliers (14.70%)
 -> Cột 'PAY_3': Phát hiện 4209 outliers (14.03%)
 -> Cột 'PAY_4': Phát hiện 3508 outliers (11.69%)
 -> Cột 'PAY_5': Phát hiện 2968 outliers (9.89%)
 -> Cột 'PAY_6': Phát hiện 3079 outliers (10.26%)
 -> Cột 'BILL_AMT1': Phát hiện 2400 outliers (8.00%)
 -> Cột 'BILL_AMT2': Phát hiện 2395 outliers (7.98%)
 -> Cột 'BILL_AMT3': Phát hiện 2469 outliers (8.23%)
 -> Cột 'BILL_AMT4': Phát hiện 2622 outliers (8.74%)
 -> Cột 'BILL_AMT5': Phát hiện 2725 outliers (9.08%)
 -> Cột 'BILL_AMT6': Phát hiện 2693 outliers (8.98%)
 -> Cột 'PAY_AMT1': Phát hiện 2745 outliers (9.15%)
 -> Cột 

In [83]:
total_rows = len(df)
# ---------------------------------------------------------
# 1. Kiểm tra cột EDUCATION
# ---------------------------------------------------------
edu_invalid_codes = [0, 4, 5, 6]
edu_invalid_count = df['EDUCATION'].isin(edu_invalid_codes).sum()
edu_invalid_pct = (edu_invalid_count / total_rows) * 100

print(f"• EDUCATION – Mã 0, 4, 5, 6: Metadata chỉ định nghĩa mã 1, 2, 3. "
      f"Có {edu_invalid_count} bản ghi ({edu_invalid_pct:.2f}%) sử dụng mã ngoài phạm vi. ")

# ---------------------------------------------------------
# 2. Kiểm tra cột MARRIAGE
# ---------------------------------------------------------
mar_invalid_count = (df['MARRIAGE'] == 0).sum()
mar_invalid_pct = (mar_invalid_count / total_rows) * 100

print(f"• MARRIAGE – Mã 0: Metadata định nghĩa mã 1, 2, 3. "
      f"Có {mar_invalid_count} bản ghi ({mar_invalid_pct:.2f}%) mang mã 0. ")

# ---------------------------------------------------------
# 3. Kiểm tra các cột PAY_* (Lấy PAY_0 làm ví dụ đại diện)
# ---------------------------------------------------------
# Cột PAY trong dataset này thường là PAY_0, PAY_2, PAY_3, PAY_4, PAY_5, PAY_6
if 'PAY_0' in df.columns:
    pay0_invalid_count = (df['PAY_0'] == -2).sum()
    pay0_invalid_pct = (pay0_invalid_count / total_rows) * 100
    
    print(f"• PAY_* – Giá trị -2: Tài liệu gốc chỉ mô tả -1 (trả đúng hạn) và 1-9 (trễ N tháng). "
          f"Giá trị -2 xuất hiện với số lượng lớn (PAY_0: {pay0_invalid_count:,} bản ghi = {pay0_invalid_pct:.1f}%). ")

• EDUCATION – Mã 0, 4, 5, 6: Metadata chỉ định nghĩa mã 1, 2, 3. Có 468 bản ghi (1.56%) sử dụng mã ngoài phạm vi. 
• MARRIAGE – Mã 0: Metadata định nghĩa mã 1, 2, 3. Có 54 bản ghi (0.18%) mang mã 0. 
• PAY_* – Giá trị -2: Tài liệu gốc chỉ mô tả -1 (trả đúng hạn) và 1-9 (trễ N tháng). Giá trị -2 xuất hiện với số lượng lớn (PAY_0: 2,759 bản ghi = 9.2%). 


In [84]:
# Xử lý cột EDUCATION: Gộp các mã 0, 4, 5, 6 thành mã 4 (đại diện cho nhóm "Khác/Unknown")
df['EDUCATION'] = df['EDUCATION'].replace([0, 4, 5, 6], 4)

# Xử lý cột MARRIAGE: Gộp mã 0 thành mã 3 (đại diện cho nhóm "Khác")
df['MARRIAGE'] = df['MARRIAGE'].replace(0, 3)

# Xác nhận lại xem dữ liệu đã sạch chưa
print("\nKiểm tra lại sau khi xử lý:")
print("- Các giá trị hiện có của EDUCATION:", df['EDUCATION'].unique().tolist())
print("- Các giá trị hiện có của MARRIAGE:", df['MARRIAGE'].unique().tolist())


Kiểm tra lại sau khi xử lý:
- Các giá trị hiện có của EDUCATION: [2, 1, 3, 4]
- Các giá trị hiện có của MARRIAGE: [1, 2, 3]


## Lưu file

In [85]:
df.to_csv("default_credit_card_cleaned_v2.csv", index=False, encoding="utf-8-sig")

# III. Vẽ biểu đồ


## 1. Tiền xử lý và tạo nhãn

In [91]:
df.rename(columns={'default payment next month': 'DEFAULT'}, inplace=True)
# Giới tính
df['SEX_LBL'] = df['SEX'].map({1: 'Nam', 2: 'Nữ'})

# Học vấn – gộp mã 0, 4, 5, 6 (không có trong metadata) vào "Khác"
edu_map = {0: 'Khác', 1: 'Sau ĐH', 2: 'Đại học', 3: 'THPT',
           4: 'Khác',  5: 'Khác',   6: 'Khác'}
df['EDU_LBL'] = df['EDUCATION'].map(edu_map)

# Hôn nhân – gộp mã 0 vào "Khác"
mar_map = {0: 'Khác', 1: 'Đã kết hôn', 2: 'Độc thân', 3: 'Khác'}
df['MAR_LBL'] = df['MARRIAGE'].map(mar_map)

# Nhóm tuổi
age_bins = [20, 25, 30, 35, 40, 50, 60, 80]
age_lbs  = ['21-25', '26-30', '31-35', '36-40', '41-50', '51-60', '60+']
df['AGE_GRP'] = pd.cut(df['AGE'], bins=age_bins, labels=age_lbs)

# Nhóm hạn mức tín dụng
lim_bins = [0, 50_000, 100_000, 200_000, 300_000, 500_000, 1_000_001]
lim_lbs  = ['<50K', '50-100K', '100-200K', '200-300K', '300-500K', '>500K']
df['LIM_GRP'] = pd.cut(df['LIMIT_BAL'], bins=lim_bins, labels=lim_lbs)

# Nhãn trạng thái thanh toán PAY_0
pay_map = {-2: 'Không dùng', -1: 'Trả đúng hạn', 0: 'Trả tối thiểu',
            1: 'Trễ 1 tháng',  2: 'Trễ 2 tháng',   3: 'Trễ 3 tháng',
            4: 'Trễ 4+ tháng'}
df['PAY_LBL'] = df['PAY_0'].map(pay_map).fillna('Trễ 4+ tháng')


## 2. Bảng màu và hàm tiện ích

In [92]:
# BẢNG MÀU ─────────────────────────────────────────────────────────────
C_BLUE  = '#2563EB'
C_RED   = '#EF4444'
C_GREEN = '#10B981'
C_AMB   = '#F59E0B'
C_PUR   = '#8B5CF6'
BG      = '#F1F5F9'   # nền figure
WHITE   = '#FFFFFF'   # nền axes

# HÀM TIỆN ÍCH ─────────────────────────────────────────────────────────
def style_ax(ax, title, xlabel='', ylabel=''):
    """Áp dụng phong cách thống nhất cho một axes."""
    ax.set_facecolor(WHITE)
    for sp in ax.spines.values():
        sp.set_color('#CBD5E1')
    ax.set_title(title, fontsize=12, fontweight='bold', color='#1E293B', pad=10)
    if xlabel:
        ax.set_xlabel(xlabel, fontsize=9, color='#64748B')
    if ylabel:
        ax.set_ylabel(ylabel, fontsize=9, color='#64748B')
    ax.tick_params(colors='#475569', labelsize=8)


def add_bar_labels(ax, fmt='{:.1f}%', pad=0.3, fontsize=9):
    """Thêm nhãn số liệu lên đỉnh mỗi cột (bar chart dọc)."""
    for bar in ax.patches:
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + pad,
            fmt.format(bar.get_height()),
            ha='center', va='bottom',
            fontsize=fontsize, fontweight='bold', color='#1E293B'
        )


def add_hbar_labels(ax, fmt='{:.1f}%', pad=0.3, fontsize=10):
    """Thêm nhãn số liệu bên phải mỗi cột (bar chart ngang)."""
    for bar in ax.patches:
        ax.text(
            bar.get_width() + pad,
            bar.get_y() + bar.get_height() / 2,
            fmt.format(bar.get_width()),
            ha='left', va='center',
            fontsize=fontsize, fontweight='bold', color='#1E293B'
        )

## 3. Biểu đồ

### 1. Phân tích tổng quan và nhân khẩu học

In [93]:
fig1, axes = plt.subplots(2, 3, figsize=(18, 11))
fig1.patch.set_facecolor(BG)
fig1.suptitle('Tổng Quan & Nhân Khẩu Học',
              fontsize=16, fontweight='bold', color='#1E293B', y=1.01)

# ── A: Donut – Tỷ lệ vỡ nợ tổng thể ────────────────────────────────────────
ax = axes[0, 0]
ax.set_facecolor(WHITE)
for sp in ax.spines.values():
    sp.set_color('#CBD5E1')

n_default    = int(df['DEFAULT'].sum())
n_no_default = len(df) - n_default
pct_default  = n_default / len(df) * 100

wedges, _ = ax.pie(
    [n_no_default, n_default],
    colors=[C_GREEN, C_RED],
    startangle=90,
    wedgeprops=dict(width=0.52)       # donut: ring width
)
ax.text(0,  0.08, f'{pct_default:.1f}%',
        ha='center', va='center', fontsize=20, fontweight='bold', color=C_RED)
ax.text(0, -0.18, 'Tỷ lệ\nvỡ nợ',
        ha='center', va='center', fontsize=10, color='#475569')

legend_els = [
    mpatches.Patch(color=C_GREEN, label=f'Không vỡ nợ ({100-pct_default:.1f}%)'),
    mpatches.Patch(color=C_RED,   label=f'Vỡ nợ ({pct_default:.1f}%)'),
]
ax.legend(handles=legend_els, loc='lower center',
          bbox_to_anchor=(0.5, -0.12), fontsize=9)
ax.set_title(f'Tỷ Lệ Vỡ Nợ Tổng Thể\n(n = {len(df):,} khách hàng)',
             fontsize=12, fontweight='bold', color='#1E293B')

# ── B: Bar – Tỷ lệ vỡ nợ theo giới tính ─────────────────────────────────────
ax = axes[0, 1]
sex_stats = df.groupby('SEX_LBL').agg(
    pct=('DEFAULT', 'mean'),
    cnt=('DEFAULT', 'count')
).reset_index()
sex_stats['pct'] *= 100

bars = ax.bar(sex_stats['SEX_LBL'], sex_stats['pct'],
              color=[C_BLUE, C_PUR], edgecolor='white', width=0.5)

# Đường trung bình chung
ax.axhline(pct_default, color='#94A3B8', ls='--', lw=1.5,
           label=f'TB chung ({pct_default:.1f}%)')
ax.legend(fontsize=8)

# Nhãn số lượng mẫu
for i, row in sex_stats.iterrows():
    ax.text(i, 1, f"n={row['cnt']:,}",
            ha='center', va='bottom', fontsize=8, color='#64748B')

add_bar_labels(ax)
style_ax(ax, 'Tỷ Lệ Vỡ Nợ Theo Giới Tính', 'Giới tính', 'Tỷ lệ vỡ nợ (%)')

# ── C: Barh – Tỷ lệ vỡ nợ theo học vấn ─────────────────────────────────────
ax = axes[0, 2]
edu_order = ['Sau ĐH', 'Đại học', 'THPT', 'Khác']
edu_pct = (df[df['EDU_LBL'].isin(edu_order)]
           .groupby('EDU_LBL')['DEFAULT']
           .mean()
           .mul(100)
           .reindex(edu_order))

bars = ax.barh(edu_pct.index, edu_pct.values,
               color=[C_GREEN, C_AMB, C_RED, C_BLUE],
               edgecolor='white', height=0.55)
ax.axvline(pct_default, color='#94A3B8', ls='--', lw=1.5,
           label=f'TB chung ({pct_default:.1f}%)')
ax.legend(fontsize=8)
add_hbar_labels(ax)
style_ax(ax, 'Tỷ Lệ Vỡ Nợ Theo Trình Độ\nHọc Vấn', 'Tỷ lệ (%)')

# ── D: Histogram – Phân phối tuổi theo nhóm vỡ nợ ──────────────────────────
ax = axes[1, 0]
ax.set_facecolor(WHITE)
for sp in ax.spines.values():
    sp.set_color('#CBD5E1')

ax.hist(df[df['DEFAULT'] == 0]['AGE'], bins=25, color=C_GREEN,
        alpha=0.6, label='Không vỡ nợ', edgecolor='white')
ax.hist(df[df['DEFAULT'] == 1]['AGE'], bins=25, color=C_RED,
        alpha=0.7, label='Vỡ nợ',       edgecolor='white')
ax.legend(fontsize=9)
style_ax(ax, 'Phân Phối Tuổi Theo Nhóm', 'Tuổi', 'Số lượng khách hàng')

# ── E: Bar – Tỷ lệ vỡ nợ theo hôn nhân ─────────────────────────────────────
ax = axes[1, 1]
mar_order = ['Đã kết hôn', 'Độc thân', 'Khác']
mar_pct = (df[df['MAR_LBL'].isin(mar_order)]
           .groupby('MAR_LBL')['DEFAULT']
           .mean()
           .mul(100)
           .reindex(mar_order))

bars = ax.bar(mar_pct.index, mar_pct.values,
              color=[C_BLUE, C_AMB, C_PUR],
              edgecolor='white', width=0.5)
ax.axhline(pct_default, color='#94A3B8', ls='--', lw=1.5)
add_bar_labels(ax)
style_ax(ax, 'Tỷ Lệ Vỡ Nợ Theo\nTình Trạng Hôn Nhân', 'Tình trạng', 'Tỷ lệ (%)')

# ── F: Bar – Tỷ lệ vỡ nợ theo nhóm tuổi ────────────────────────────────────
ax = axes[1, 2]
age_pct = df.groupby('AGE_GRP', observed=True)['DEFAULT'].mean().mul(100)

clrs = [C_RED if v > pct_default else C_BLUE for v in age_pct.values]
x = np.arange(len(age_lbs))
ax.bar(x, age_pct.values, color=clrs, edgecolor='white', width=0.65)
ax.set_xticks(x)
ax.set_xticklabels(age_lbs, rotation=30, fontsize=8)
ax.axhline(pct_default, color='#94A3B8', ls='--', lw=1.5,
           label=f'TB chung ({pct_default:.1f}%)')
ax.legend(fontsize=8)

for i, v in enumerate(age_pct.values):
    ax.text(i, v + 0.2, f'{v:.1f}%',
            ha='center', va='bottom', fontsize=8, fontweight='bold')

style_ax(ax, 'Tỷ Lệ Vỡ Nợ Theo Nhóm Tuổi', 'Nhóm tuổi', 'Tỷ lệ (%)')

plt.tight_layout(pad=2.5)
plt.savefig('fig1_demographics.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.close()

### 2. Hạn mức tín dụng và hành vi thanh toán

In [94]:
fig2, axes = plt.subplots(2, 3, figsize=(18, 11))
fig2.patch.set_facecolor(BG)
fig2.suptitle('Hạn Mức Tín Dụng & Hành Vi Thanh Toán',
              fontsize=16, fontweight='bold', color='#1E293B', y=1.01)

# ── A: Bar – Tỷ lệ vỡ nợ theo hạn mức tín dụng ─────────────────────────────
ax = axes[0, 0]
lim_pct = df.groupby('LIM_GRP', observed=True)['DEFAULT'].mean().mul(100)

clrs = [C_RED if v > pct_default else C_BLUE for v in lim_pct.values]
bars = ax.bar(lim_pct.index, lim_pct.values,
              color=clrs, edgecolor='white', width=0.65)
ax.axhline(pct_default, color='#94A3B8', ls='--', lw=1.5,
           label=f'TB chung ({pct_default:.1f}%)')
ax.legend(fontsize=8)
ax.tick_params(axis='x', rotation=35)

for bar in bars:
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.3,
            f'{bar.get_height():.1f}%',
            ha='center', va='bottom', fontsize=9, fontweight='bold')

style_ax(ax, 'Tỷ Lệ Vỡ Nợ Theo\nHạn Mức Tín Dụng (NT$)',
         'Nhóm hạn mức', 'Tỷ lệ (%)')

# ── B: Histogram – Phân phối LIMIT_BAL theo nhóm vỡ nợ ─────────────────────
ax = axes[0, 1]
ax.set_facecolor(WHITE)
for sp in ax.spines.values():
    sp.set_color('#CBD5E1')

ax.hist(df[df['DEFAULT'] == 0]['LIMIT_BAL'], bins=30,
        color=C_GREEN, alpha=0.6, label='Không vỡ nợ', edgecolor='white')
ax.hist(df[df['DEFAULT'] == 1]['LIMIT_BAL'], bins=30,
        color=C_RED,   alpha=0.7, label='Vỡ nợ',       edgecolor='white')
ax.legend(fontsize=9)
style_ax(ax, 'Phân Phối Hạn Mức Tín Dụng',
         'Hạn mức (NT$)', 'Số lượng khách hàng')

# ── C: Bar – Tỷ lệ vỡ nợ theo trạng thái thanh toán PAY_0 ──────────────────
ax = axes[0, 2]
ord_pay = ['Không dùng', 'Trả đúng hạn', 'Trả tối thiểu',
           'Trễ 1 tháng', 'Trễ 2 tháng', 'Trễ 3 tháng', 'Trễ 4+ tháng']
pay_pct = (df.groupby('PAY_LBL')['DEFAULT']
            .mean()
            .mul(100)
            .reindex([x for x in ord_pay if x in df['PAY_LBL'].unique()]))

bar_cols = [C_GREEN if v < 20 else C_AMB if v < 50 else C_RED
            for v in pay_pct.values]
bars = ax.bar(range(len(pay_pct)), pay_pct.values,
              color=bar_cols, edgecolor='white', width=0.65)
ax.set_xticks(range(len(pay_pct)))
ax.set_xticklabels(pay_pct.index, rotation=35, ha='right', fontsize=8)
ax.axhline(pct_default, color='#94A3B8', ls='--', lw=1.5)

for bar in bars:
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.8,
            f'{bar.get_height():.0f}%',
            ha='center', va='bottom', fontsize=9, fontweight='bold')

style_ax(ax, 'Tỷ Lệ Vỡ Nợ Theo\nTrạng Thái TT Tháng 9/2005',
         'Trạng thái thanh toán', 'Tỷ lệ (%)')

# ── D: Line – Xu hướng dư nợ trung bình 6 tháng ────────────────────────────
ax = axes[1, 0]
ax.set_facecolor(WHITE)
for sp in ax.spines.values():
    sp.set_color('#CBD5E1')

months    = ['T4/05', 'T5/05', 'T6/05', 'T7/05', 'T8/05', 'T9/05']
bill_cols = ['BILL_AMT6', 'BILL_AMT5', 'BILL_AMT4',
             'BILL_AMT3', 'BILL_AMT2', 'BILL_AMT1']

avg_bill_d = [df[df['DEFAULT'] == 1][c].mean() / 1000 for c in bill_cols]
avg_bill_n = [df[df['DEFAULT'] == 0][c].mean() / 1000 for c in bill_cols]

ax.plot(months, avg_bill_d, marker='o', color=C_RED,
        lw=2.5, ms=7, label='Vỡ nợ')
ax.plot(months, avg_bill_n, marker='s', color=C_GREEN,
        lw=2.5, ms=7, label='Không vỡ nợ')
ax.fill_between(months, avg_bill_d, avg_bill_n,
                alpha=0.08, color=C_RED)
ax.legend(fontsize=9)
ax.grid(axis='y', ls='--', alpha=0.4)
style_ax(ax, 'Dư Nợ Trung Bình 6 Tháng',
         'Tháng', 'Dư nợ TB (NT$ nghìn)')

# ── E: Line – Xu hướng số tiền trả trung bình 6 tháng ──────────────────────
ax = axes[1, 1]
ax.set_facecolor(WHITE)
for sp in ax.spines.values():
    sp.set_color('#CBD5E1')

pay_cols  = ['PAY_AMT6', 'PAY_AMT5', 'PAY_AMT4',
             'PAY_AMT3', 'PAY_AMT2', 'PAY_AMT1']

avg_pay_d = [df[df['DEFAULT'] == 1][c].mean() / 1000 for c in pay_cols]
avg_pay_n = [df[df['DEFAULT'] == 0][c].mean() / 1000 for c in pay_cols]

ax.plot(months, avg_pay_d, marker='o', color=C_RED,
        lw=2.5, ms=7, label='Vỡ nợ')
ax.plot(months, avg_pay_n, marker='s', color=C_GREEN,
        lw=2.5, ms=7, label='Không vỡ nợ')
ax.legend(fontsize=9)
ax.grid(axis='y', ls='--', alpha=0.4)
style_ax(ax, 'Số Tiền Trả Trung Bình 6 Tháng',
         'Tháng', 'Số tiền TB (NT$ nghìn)')

# ── F: Heatmap – Ma trận tương quan các biến chính ──────────────────────────
ax = axes[1, 2]
key_cols = ['LIMIT_BAL', 'AGE', 'PAY_0', 'PAY_2',
            'BILL_AMT1', 'PAY_AMT1', 'DEFAULT']
nice_names = {
    'LIMIT_BAL': 'Hạn mức',
    'AGE':       'Tuổi',
    'PAY_0':     'TT T9',
    'PAY_2':     'TT T8',
    'BILL_AMT1': 'Dư nợ T9',
    'PAY_AMT1':  'Trả nợ T9',
    'DEFAULT':   'Vỡ nợ',
}
corr = df[key_cols].corr()
corr.index   = [nice_names[c] for c in key_cols]
corr.columns = [nice_names[c] for c in key_cols]

sns.heatmap(corr, ax=ax, annot=True, fmt='.2f',
            cmap='RdBu_r', center=0,
            linewidths=0.5,
            annot_kws={'size': 9, 'weight': 'bold'},
            cbar_kws={'shrink': 0.8})
ax.set_title('Tương Quan Giữa Các Biến Chính',
             fontsize=12, fontweight='bold', color='#1E293B', pad=10)
ax.tick_params(axis='x', rotation=35, labelsize=8)
ax.tick_params(axis='y', rotation=0,  labelsize=8)

plt.tight_layout(pad=2.5)
plt.savefig('fig2_credit_payment.png', dpi=150,
            bbox_inches='tight', facecolor=BG)
plt.close()


### 3. Chất Lượng Dữ Liệu & Kết Quả Mô Hình

In [95]:
# ── Huấn luyện Logistic Regression ───────────────────────────────────────────
features = ['LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE',
            'PAY_0',     'PAY_2',    'PAY_3',
            'BILL_AMT1', 'BILL_AMT2', 'PAY_AMT1', 'PAY_AMT2']

X = df[features].values
y = df['DEFAULT'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler  = StandardScaler()
X_tr_s  = scaler.fit_transform(X_train)
X_te_s  = scaler.transform(X_test)

lr = LogisticRegression(max_iter=500, random_state=42)
lr.fit(X_tr_s, y_train)

y_pred = lr.predict(X_te_s)
y_prob = lr.predict_proba(X_te_s)[:, 1]
fpr, tpr, _ = roc_curve(y_test, y_prob)
auc          = roc_auc_score(y_test, y_prob)
rpt          = classification_report(y_test, y_pred, output_dict=True)

print(f"  AUC-ROC  : {auc:.4f}")
print(f"  Accuracy : {rpt['accuracy']:.4f}")
print(f"  Precision: {rpt['1']['precision']:.4f}")
print(f"  Recall   : {rpt['1']['recall']:.4f}")
print(f"  F1-Score : {rpt['1']['f1-score']:.4f}")

# ── Vẽ ───────────────────────────────────────────────────────────────────────
fig3, axes = plt.subplots(2, 3, figsize=(18, 11))
fig3.patch.set_facecolor(BG)
fig3.suptitle('Chất Lượng Dữ Liệu & Mô Hình Dự Đoán',
              fontsize=16, fontweight='bold', color='#1E293B', y=1.01)

# ── A: Bar – Thống kê các điểm bất ổn dữ liệu ───────────────────────────────
ax = axes[0, 0]
ax.set_facecolor(WHITE)
for sp in ax.spines.values():
    sp.set_color('#CBD5E1')

issues = {
    'EDUCATION\n(mã 0,4,5,6)': int((~df['EDUCATION'].isin([1, 2, 3])).sum()),
    'MARRIAGE\n(mã 0)':        int((df['MARRIAGE'] == 0).sum()),
    'PAY_*\n(mã -2)':          int(sum((df[c] == -2).sum()
                                       for c in ['PAY_0','PAY_2','PAY_3',
                                                 'PAY_4','PAY_5','PAY_6'])),
    'BILL_AMT\nâm':            int(sum((df[f'BILL_AMT{i}'] < 0).sum()
                                       for i in range(1, 7))),
}
bars = ax.bar(issues.keys(), issues.values(),
              color=[C_RED, C_AMB, C_AMB, C_AMB],
              edgecolor='white', width=0.55)

for bar in bars:
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 10,
            f'{int(bar.get_height()):,}',
            ha='center', va='bottom', fontsize=11, fontweight='bold',
            color='#1E293B')

ax.set_title('Các Điểm Bất Ổn Trong Dữ Liệu',
             fontsize=12, fontweight='bold', color='#1E293B', pad=10)
ax.set_ylabel('Số lượng bản ghi bị ảnh hưởng', fontsize=9, color='#64748B')
ax.tick_params(colors='#475569', labelsize=9)

# ── B: Grouped Bar – Phân bổ mã PAY -2 vs -1 theo từng tháng ───────────────
ax = axes[0, 1]
ax.set_facecolor(WHITE)
for sp in ax.spines.values():
    sp.set_color('#CBD5E1')

pay_cols_list = ['PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']
month_lbs     = ['T9/05', 'T8/05', 'T7/05', 'T6/05', 'T5/05', 'T4/05']

neg2_counts = [(df[c] == -2).sum() for c in pay_cols_list]
neg1_counts = [(df[c] == -1).sum() for c in pay_cols_list]

x = np.arange(len(month_lbs))
w = 0.38
ax.bar(x - w / 2, neg2_counts, w, color=C_RED,
       label='Mã -2 (không có trong metadata)', edgecolor='white')
ax.bar(x + w / 2, neg1_counts, w, color=C_BLUE,
       label='Mã -1 (trả đúng hạn)',             edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels(month_lbs)
ax.legend(fontsize=8)
ax.set_title('Phân Bổ Giá Trị PAY = -2 vs -1\nTheo Từng Tháng',
             fontsize=12, fontweight='bold', color='#1E293B', pad=10)
ax.set_ylabel('Số lượng', fontsize=9, color='#64748B')
ax.tick_params(colors='#475569', labelsize=9)

# ──C: Boxplot – Dư nợ âm phân theo tháng ───────────────────────────────────
ax = axes[0, 2]
ax.set_facecolor(WHITE)
for sp in ax.spines.values():
    sp.set_color('#CBD5E1')

neg_bills = [df[df[f'BILL_AMT{i}'] < 0][f'BILL_AMT{i}'].values / 1000
             for i in range(1, 7)]
bill_month_lbs = ['T9', 'T8', 'T7', 'T6', 'T5', 'T4']

bp = ax.boxplot(neg_bills, tick_labels=bill_month_lbs,
                patch_artist=True,
                medianprops=dict(color='white', lw=2))
for patch in bp['boxes']:
    patch.set_facecolor(C_AMB)
    patch.set_alpha(0.7)

ax.axhline(0, color=C_RED, ls='--', lw=1.5, label='Ngưỡng 0')
ax.legend(fontsize=8)
ax.set_title('Phân Phối Dư Nợ Âm Theo Tháng\n(NT$ nghìn)',
             fontsize=12, fontweight='bold', color='#1E293B', pad=10)
ax.set_ylabel('Dư nợ (NT$ nghìn)', fontsize=9, color='#64748B')
ax.tick_params(colors='#475569', labelsize=9)

# ──D: Line – Đường cong ROC ─────────────────────────────────────────────────
ax = axes[1, 0]
ax.set_facecolor(WHITE)
for sp in ax.spines.values():
    sp.set_color('#CBD5E1')

ax.plot(fpr, tpr, color=C_BLUE, lw=2.5,
        label=f'Logistic Regression (AUC = {auc:.3f})')
ax.plot([0, 1], [0, 1], '--', color='#94A3B8', lw=1.5,
        label='Random (AUC = 0.500)')
ax.fill_between(fpr, tpr, alpha=0.08, color=C_BLUE)
ax.legend(fontsize=9)
ax.grid(ls='--', alpha=0.3)
style_ax(ax, 'Đường Cong ROC\nMô Hình Logistic Regression',
         'False Positive Rate', 'True Positive Rate')

# ──E: Barh – Hệ số tác động (Feature Importance) ───────────────────────────
ax = axes[1, 1]
ax.set_facecolor(WHITE)
for sp in ax.spines.values():
    sp.set_color('#CBD5E1')

nice = {
    'LIMIT_BAL':  'Hạn mức tín dụng',
    'SEX':        'Giới tính',
    'EDUCATION':  'Học vấn',
    'MARRIAGE':   'Hôn nhân',
    'AGE':        'Tuổi',
    'PAY_0':      'Trạng thái TT T9',
    'PAY_2':      'Trạng thái TT T8',
    'PAY_3':      'Trạng thái TT T7',
    'BILL_AMT1':  'Dư nợ T9',
    'BILL_AMT2':  'Dư nợ T8',
    'PAY_AMT1':   'Số tiền trả T9',
    'PAY_AMT2':   'Số tiền trả T8',
}
coef = pd.Series(lr.coef_[0], index=features).sort_values()
coef.index = [nice[i] for i in coef.index]

clrs_coef = [C_RED if v > 0 else C_GREEN for v in coef.values]
coef.plot(kind='barh', ax=ax, color=clrs_coef, edgecolor='white')
ax.axvline(0, color='#94A3B8', lw=1.5)

legend_els = [
    mpatches.Patch(color=C_RED,   label='Tăng rủi ro vỡ nợ'),
    mpatches.Patch(color=C_GREEN, label='Giảm rủi ro vỡ nợ'),
]
ax.legend(handles=legend_els, fontsize=8)
ax.set_title('Hệ Số Tác Động Từng Biến\n(Logistic Regression – sau chuẩn hóa)',
             fontsize=12, fontweight='bold', color='#1E293B', pad=10)
ax.set_xlabel('Hệ số (coefficient)', fontsize=9, color='#64748B')
ax.tick_params(colors='#475569', labelsize=8)

# ──F: Heatmap – Confusion Matrix ────────────────────────────────────────────
ax = axes[1, 2]
ax.set_facecolor(WHITE)
for sp in ax.spines.values():
    sp.set_color('#CBD5E1')

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(
    cm, annot=True, fmt='d',
    cmap='Blues', ax=ax,
    xticklabels=['Dự đoán: 0', 'Dự đoán: 1'],
    yticklabels=['Thực tế: 0',  'Thực tế: 1'],
    linewidths=1,
    annot_kws={'size': 14, 'weight': 'bold'},
    cbar=False
)
ax.set_title(
    f'Ma Trận Nhầm Lẫn (Confusion Matrix)\n'
    f'Accuracy={rpt["accuracy"]:.3f} | AUC={auc:.3f} | '
    f'F1={rpt["1"]["f1-score"]:.3f}',
    fontsize=11, fontweight='bold', color='#1E293B', pad=10
)
ax.tick_params(colors='#475569', labelsize=9)

plt.tight_layout(pad=2.5)
plt.savefig('fig3_model.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.close()


  AUC-ROC  : 0.7073
  Accuracy : 0.8087
  Precision: 0.6950
  Recall   : 0.2404
  F1-Score : 0.3572
